# Petrophysical Interpretation from LAS Files

This tutorial demonstrates a complete petrophysical workflow starting from LAS file loading through to net pay analysis.

## What you'll learn

- Loading well data from LAS files
- Inspecting available curves and data quality
- Setting up interpretation parameters
- Running a complete petrophysical interpretation
- Using the built-in `plot()` method for visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import welly
from welly import Well
from welly.petro import (
    PetroInterpreter,
    PetrophysicalParameters,
    MatrixParameters,
    FluidParameters,
    ClayParameters,
)

print(f"welly version: {welly.__version__}")

## 1. Loading Well Data from LAS

Use `Well.from_las()` to load a well from a LAS file.

In [ ]:
# Load well from LAS file
well = Well.from_las('data/P-129_out.LAS')

print(f"Well name: {well.name}")
print(f"\nAvailable curves ({len(well.data)}):")
for name in sorted(well.data.keys()):
    curve = well.data[name]
    print(f"  {name}: {len(curve)} samples")

## 2. Quick Look at the Data

Plot the key input curves to understand the data.

In [ ]:
# Get depth index
depth = well.data['GR'].df.index

fig, axes = plt.subplots(1, 4, figsize=(14, 10), sharey=True)

# GR
axes[0].plot(well.data['GR'].values, depth, 'g-', lw=0.5)
axes[0].set_xlabel('GR (API)')
axes[0].set_ylabel('Depth (m)')
axes[0].set_xlim(0, 150)

# Density
axes[1].plot(well.data['RHOB'].values, depth, 'r-', lw=0.5)
axes[1].set_xlabel('RHOB (g/cc)')
axes[1].set_xlim(1.8, 2.8)

# Neutron (use limestone matrix)
if 'NPHI_LIM' in well.data:
    axes[2].plot(well.data['NPHI_LIM'].values, depth, 'b-', lw=0.5)
    axes[2].set_xlabel('NPHI (v/v)')
    axes[2].set_xlim(0.45, -0.05)

# Resistivity
if 'RT_HRLT' in well.data:
    axes[3].plot(well.data['RT_HRLT'].values, depth, 'k-', lw=0.5)
    axes[3].set_xlabel('RT (ohm.m)')
    axes[3].set_xscale('log')

for ax in axes:
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Input Logs: {well.name}')
plt.tight_layout()
plt.show()

## 3. Set Up Interpretation Parameters

Define the petrophysical parameters for this formation.

In [ ]:
# Examine GR to pick clean/shale values
gr = well.data['GR'].values
print(f"GR statistics:")
print(f"  Min: {np.nanmin(gr):.1f} API")
print(f"  Max: {np.nanmax(gr):.1f} API")
print(f"  P10: {np.nanpercentile(gr, 10):.1f} API")
print(f"  P90: {np.nanpercentile(gr, 90):.1f} API")

In [ ]:
# Create parameters based on log statistics
params = PetrophysicalParameters(
    matrix=MatrixParameters.sandstone(),
    fluid=FluidParameters(rw=0.05, rw_temp=75),
    clay=ClayParameters(
        gr_clean=25,
        gr_shale=130,
        nphi_shale=0.35,
        rho_shale=2.55,
        rt_shale=5.0
    ),
    a=0.81,
    m=2.0,
    n=2.0,
    name='Kennetcook Sandstone'
)

print(f"Parameters: {params.name}")
print(f"  Matrix: {params.matrix.lithology}")
print(f"  Rw: {params.fluid.rw} ohm.m")
print(f"  GR clean/shale: {params.clay.gr_clean}/{params.clay.gr_shale}")

## 4. Create Interpreter and Check Inputs

In [ ]:
# Create interpreter using the well.petro() convenience method
interp = well.petro(params=params)

# Check what curves are available and what calculations are possible
status = interp.check_inputs(verbose=True)

In [ ]:
# See alias matches - which actual curve names map to standard names
matches = interp.get_alias_matches()
print("\nCurve alias matches:")
for std, actual in matches.items():
    if actual:
        print(f"  {std} -> {actual}")

## 5. Run Interpretation

In [ ]:
# Run standard interpretation workflow
results = interp.run_standard_interpretation(
    vshale_method='larionov',
    porosity_method='density',
    sw_method='archie',
    phi_cutoff=0.08,
    sw_cutoff=0.50,
    vsh_cutoff=0.40
)

print("Curves computed:", results['curves_computed'])
if results['warnings']:
    print("Warnings:", results['warnings'])
print("\nStatistics:")
for k, v in results['statistics'].items():
    print(f"  {k}: {v:.4f}")

## 6. Visualize Results with interp.plot()

The `PetroInterpreter` has a built-in `plot()` method that creates a standard multi-track interpretation plot.

In [ ]:
# Default plot - auto-selects tracks based on available curves
fig = interp.plot()
plt.show()

In [ ]:
# Plot with custom depth range
fig = interp.plot(depth_range=(1500, 1800))
plt.show()

In [ ]:
# Plot only results (no input curves)
fig = interp.plot(show_inputs=False, title='Interpretation Results Only')
plt.show()

In [ ]:
# Custom track selection
fig = interp.plot(tracks=['GR', 'VSH', 'PHI', 'SW', 'PAY'])
plt.show()

## 7. Save the Plot

The `plot()` method returns a matplotlib Figure object that can be saved.

In [ ]:
# Create and save a publication-quality figure
fig = interp.plot(figsize=(16, 12), title=f'Petrophysical Interpretation: {well.name}')
fig.savefig('data/interpretation_results.png', dpi=150, bbox_inches='tight')
print("Saved: data/interpretation_results.png")
plt.show()

## 8. Summary Statistics

In [ ]:
# Get interpretation summary
summary = interp.summary()

print(f"Interpretation Summary for {summary['well_name']}")
print("=" * 50)
print(f"Parameters used: {summary['parameters']}")
print(f"Curves computed: {summary['curves_computed']}")
print("\nCurve statistics:")
for curve, stats in summary['statistics'].items():
    print(f"  {curve}:")
    print(f"    Mean: {stats['mean']:.4f}")
    print(f"    Range: {stats['min']:.4f} - {stats['max']:.4f}")

## 9. Export Results to LAS

The computed curves are stored in the Well object and can be exported.

In [ ]:
# Check what curves are now in the well
print("All curves in well:")
for name in sorted(well.data.keys()):
    print(f"  {name}")

In [ ]:
# Export to LAS (uncomment to run)
# well.to_las('data/P-129_interpreted.las')
# print("Exported to: data/P-129_interpreted.las")